# Various Plots to analyse influence of rain

In [1]:
from tuecycle.utils.transforms import (
    add_time_features,      # Adds hour, dayofweek, month, year_month, is_weekend
    filter_daytime,         # Filters to hours 6-22
    compute_deviations,     # Adds temp_deviation and bike_deviation columns
    classify_time_category, # Adds time_category (rush hour classification)
    add_season,             # Adds season column (Winter/Transition/Summer)
    prepare_fft_data,       # Prepares data for FFT analysis
)
from tuecycle import DataManager
import pandas as pd
from tuecycle.config.stations import get_stations_by_city
from tuecycle.plots import get_plot
from typing import Union, Sequence


1. Set the time window and cities

In [2]:
START_DATE = pd.Timestamp(2020, 1, 1)
END_DATE   = pd.Timestamp(2025, 11, 30)

dm = DataManager(
    start_date=(START_DATE.year, START_DATE.month, START_DATE.day),
    end_date=(END_DATE.year, END_DATE.month, END_DATE.day),
)

cities = ["mannheim", "tuebingen", "heidelberg"]



### 1. Preprocessing

Filter out stations that did not exist at the time of the start date. Preprocess and add necessary features

In [3]:
data_dict = {}
city_data_dict = {}

for city in cities:
    stations = get_stations_by_city(city)
    dfs_city = []

    for station in stations:
        # Does the data exist at all or is it faulty?
        try:
            df = dm.get(station.alias)
        except FileNotFoundError:
            print(f"Skipping {station.alias}, bike or weather data missing.")
            continue
        except Exception as e:
            print(f"Skipping {station.alias}, error: {e}")
            continue

        # Does the data exist for the whole time window?
        if df['datetime'].max() < START_DATE:
            print(f"Skipping {station.alias}, no data since {START_DATE.date()}")
            continue

        # Add time features
        df = add_time_features(df)
        df = classify_time_category(df)
        
        data_dict[station.alias] = df
        dfs_city.append(df)

    if dfs_city:
        city_data_dict[city] = pd.concat(dfs_city, ignore_index=True)

Loading data for Mannheim (Feudenheimstr. stadtauswärts)...
Skipping mannheim_feudenheimstr_aufwaerts, bike or weather data missing.
Loading data for Mannheim (Feudenheimstr. stadteinwärts)...
Skipping mannheim_feudenheimstr_einwaerts, bike or weather data missing.
Loading data for Mannheim (Luzenbergstr.)...
Skipping mannheim_luzenbergstr, bike or weather data missing.
Skipping mannheim_B38, error: "Unknown station 'mannheim_B38'. Available: freiburg_dreisam, freiburg_eschholz, freiburg_gueterbahn, freiburg_wiwili, heidelberg_berliner, heidelberg_eppelheimer, heidelberg_ernst_walz, heidelberg_gaisberg, heidelberg_kurfuersten, heidelberg_liebermann, heidelberg_mannheimer, heidelberg_ploeck, heidelberg_rohrbacher, heidelberg_schlierbacher, heidelberg_theodor_heuss, heidelberg_ziegelhaeuser, heilbronn_neckarufer, heilbronn_nord, heilbronn_sued, karlsruhe_erbprinzen, kirchheim_barometer, konstanz_herose, loerrach_berliner, loerrach_friedhof, ludwigsburg_alleen, ludwigsburg_favorite, ludwi

### 2. Define several functions for plotting

In [7]:
def compute_city_rain_shares(city_data_dict, temp_max, temp_min, hours: Union[str, Sequence[str], None] = None):
    """
    Calculates the percentage of bicycle journeys in rainy weather per city during rush hour.

    Args:
        city_data_dict: dict {city_name: DataFrame of all stations in given city}
        temp_max: max. Temperatur [°C]
        temp_min: min. Temperatur [°C]
        hours:  - 'Morning Rush (7-9)': Weekday hours 7-8
                - 'Evening Rush (17-19)': Weekday hours 17-18
                - 'Weekday Non-Rush': Other weekday hours
                - 'Weekend': Saturday and Sunday

    Returns:
        pd.DataFrame with ["city", "rain_share"]
    """
    records = []

    for city, df in city_data_dict.items():
        # Filters for the given temperature and rush hour
        df_analysis = df[
            (df['temp'] <= temp_max) &
            (df['temp'] >= temp_min)
        ]

        if hours is not None:
            if isinstance(hours, str):
                hours = [hours]  # convert single string to list
            df_analysis = df_analysis[df_analysis['time_category'].isin(hours)]

        if df_analysis.empty or df_analysis['bike'].sum() == 0:
            continue

        total_bikes = df_analysis['bike'].sum()
        rain_bikes = df_analysis[df_analysis['rain'] > 0]['bike'].sum()
        rain_share = rain_bikes / total_bikes

        records.append({
            "city": city.capitalize(),
            "rain_share": rain_share
        })

    return pd.DataFrame(records)


In [8]:
def filter_rain(base_data: dict, temp_max: float = 50, temp_min: float = -30,
                     hours: Union[str, Sequence[str], None] = None
) -> dict:
    """
    Filters bike traffic data for rainy conditions during rush hours and normalizes bike counts.

    Args:
        base_data (dict)
        temp_max (float, optional): max. temperature [°C]
        temp_min (float, optional): min. temperature [°C]
        hours:  - 'Morning Rush (7-9)': Weekday hours 7-8
                - 'Evening Rush (17-19)': Weekday hours 17-18
                - 'Weekday Non-Rush': Other weekday hours
                - 'Weekend': Saturday and Sunday

    Returns:
        dict: Filtered dictionary with bike counts normalized.
    """
    filtered = {}

    for alias, df in base_data.items():
        df = df.copy()

        # Min-Max Normalization of bike counts per station

        bike_min = df['bike'].min()
        bike_max = df['bike'].max()
        if bike_max > bike_min:
            df['bike'] = (df['bike'] - bike_min) / (bike_max - bike_min)
        else:
            df['bike'] = 0.0  # If all counts are the same, set normalized value to 0

        # Filter data for analysis
        df_analysis = df[
            (df['temp'] <= temp_max) &
            (df['temp'] >= temp_min) &
            (df['rain'] > 0) 
        ]

        if hours is not None:
            if isinstance(hours, str):
                hours = [hours]  # convert single string to list
            df_analysis = df_analysis[df_analysis['time_category'].isin(hours)]



        if not df_analysis.empty:
            filtered[alias] = df_analysis

    return filtered


### 6. Plot using the filter functions

- Figure 1 - 3 : Number of cyclists in the rain at different temperature (2019-2025) during rush hour  
  Set `rush_hours=None` to not filter for rush hour. 

- Figure 4: Share of cyclists in the rain for each city at ≤5°C (2019-2025) during rush hour

In [9]:
fig1 = get_plot("bike_vs_rain_rush_hour_city")(
    filter_rain(data_dict, temp_max = 5),
    cities=cities,
    title="Number of cyclists in the rain at ≤5°C (2019-2025) - Rush Hour"
)
fig1.show()

fig2 = get_plot("bike_vs_rain_rush_hour_city")(
    filter_rain(data_dict, temp_max = 40, hours=['Weekend']),
    cities=cities,
    title="Number of cyclists in the rain at ≤5°C (2019-2025) - Weekend"
)
fig2.show()

fig4 = get_plot("city_rain_share_boxplot")(
    compute_city_rain_shares(city_data_dict, temp_max=5, temp_min=-30),
    title="Rain share ≤5°C, Rush Hour"
)
fig4.show()

